# CoStrata experiments (accuracy matrix, ablations, attribute scaling)

**Before running:** `Runtime -> Change runtime type -> CPU`, and enable **High-RAM** (gives 8 vCPUs; GPUs do not help, the
workload is single-threaded C++ run as parallel jobs). Then `Runtime -> Run all`.

* Results are mirrored to `MyDrive/costrata_results/` every two minutes, and the run resumes from there if the
  runtime is recycled (just *Run all* again).
* Jobs already finished on the laptop are skipped if the regenerated data is byte-identical (checked by MD5).
* Update-throughput timings are **not** measured here (shared VMs are too noisy); they are run locally.

In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/costrata'
DRIVE = '/content/drive/MyDrive/costrata_results'
for d in ('src', 'scripts', 'colab', 'data/raw', 'work', 'results'):
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)
os.makedirs(f'{DRIVE}/results', exist_ok=True)
os.chdir(ROOT)
print('vCPUs:', os.cpu_count())
print(subprocess.run("grep -m1 'model name' /proc/cpuinfo; free -g | sed -n 2p", shell=True, capture_output=True, text=True).stdout)

In [ ]:
%%writefile Makefile
CXX ?= g++
CXXFLAGS ?= -O3 -march=native -std=c++17 -Wall -Wextra
BIN = bin/bench bin/gen_synth bin/mkworkload bin/pcap2bin

all: $(BIN)

bin/%: src/%.cpp src/common.hpp src/synopses.hpp
	@mkdir -p bin
	$(CXX) $(CXXFLAGS) -o $@ $<

clean:
	rm -rf bin


In [ ]:
%%writefile src/common.hpp
// Common utilities: hashing, dataset I/O, query representation.
#pragma once
#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <limits>
#include <string>
#include <vector>

// ---------------------------------------------------------------- hashing
static inline uint64_t mix64(uint64_t x) {
    x ^= x >> 33;
    x *= 0xff51afd7ed558ccdULL;
    x ^= x >> 33;
    x *= 0xc4ceb9fe1a85ec53ULL;
    x ^= x >> 33;
    return x;
}
// Hash a 32-bit attribute value into [0, w).
static inline uint32_t bucket_of(uint32_t v, uint64_t seed, uint32_t w) {
    uint64_t h = mix64((uint64_t)v * 0x9E3779B97F4A7C15ULL + seed);
    return (uint32_t)(((h >> 32) * (uint64_t)w) >> 32);
}
// Uniform double in [0,1) from a 64-bit hash.
static inline double to_unit(uint64_t h) { return (double)(h >> 11) * (1.0 / 9007199254740992.0); }

struct Rng { // splitmix64
    uint64_t s;
    explicit Rng(uint64_t seed) : s(seed) {}
    uint64_t next() {
        uint64_t z = (s += 0x9E3779B97F4A7C15ULL);
        z = (z ^ (z >> 30)) * 0xBF58476D1CE4E5B9ULL;
        z = (z ^ (z >> 27)) * 0x94D049BB133111EBULL;
        return z ^ (z >> 31);
    }
    double unit() { return to_unit(next()); }
    uint64_t below(uint64_t n) { return (uint64_t)(((unsigned __int128)next() * n) >> 64); }
};

// ---------------------------------------------------------------- dataset
// Binary layout: "STRM" | u32 p | u64 n | n*p u32 (row-major).
struct Dataset {
    uint64_t n = 0;
    uint32_t p = 0;
    std::vector<uint32_t> v;
    const uint32_t* row(uint64_t i) const { return v.data() + i * p; }
    bool load(const std::string& path) {
        FILE* f = fopen(path.c_str(), "rb");
        if (!f) return false;
        char magic[4];
        if (fread(magic, 1, 4, f) != 4 || memcmp(magic, "STRM", 4) != 0) { fclose(f); return false; }
        if (fread(&p, 4, 1, f) != 1 || fread(&n, 8, 1, f) != 1) { fclose(f); return false; }
        v.resize(n * p);
        size_t got = fread(v.data(), 4, n * p, f);
        fclose(f);
        return got == n * p;
    }
    bool save(const std::string& path) const {
        FILE* f = fopen(path.c_str(), "wb");
        if (!f) return false;
        fwrite("STRM", 1, 4, f);
        fwrite(&p, 4, 1, f);
        fwrite(&n, 8, 1, f);
        fwrite(v.data(), 4, n * p, f);
        fclose(f);
        return true;
    }
    // Keep only the first k attributes (used for the attribute-scaling experiment).
    void project_prefix(uint32_t k) {
        if (k >= p) return;
        std::vector<uint32_t> nv(n * k);
        for (uint64_t i = 0; i < n; i++)
            for (uint32_t j = 0; j < k; j++) nv[i * k + j] = v[i * p + j];
        v.swap(nv);
        p = k;
    }
};

// ---------------------------------------------------------------- queries
// Conjunction of equality predicates: attrs[i] == vals[i].
struct Query {
    std::vector<uint32_t> attrs, vals;
    uint64_t truth = 0;
    uint64_t minfreq = 0; // frequency of the rarest single predicate
    int kind = 0; // workload family
    bool matches(const uint32_t* rec) const {
        for (size_t i = 0; i < attrs.size(); i++)
            if (rec[attrs[i]] != vals[i]) return false;
        return true;
    }
};

// Text format, one query per line: kind npred truth minfreq a1 v1 a2 v2 ...
static inline bool load_workload(const std::string& path, std::vector<Query>& qs) {
    FILE* f = fopen(path.c_str(), "r");
    if (!f) return false;
    int kind, np;
    unsigned long long truth, minfreq;
    while (fscanf(f, "%d %d %llu %llu", &kind, &np, &truth, &minfreq) == 4) {
        Query q;
        q.kind = kind;
        q.truth = truth;
        q.minfreq = minfreq;
        for (int i = 0; i < np; i++) {
            unsigned a, v;
            if (fscanf(f, "%u %u", &a, &v) != 2) { fclose(f); return false; }
            q.attrs.push_back(a);
            q.vals.push_back(v);
        }
        qs.push_back(std::move(q));
    }
    fclose(f);
    return true;
}

static inline double now_sec() {
    return std::chrono::duration<double>(std::chrono::steady_clock::now().time_since_epoch()).count();
}


In [ ]:
%%writefile src/synopses.hpp
// Stream synopses for COUNT queries with ad-hoc conjunctive predicates.
//   Reservoir  : uniform row sample (Vitter's Algorithm L)
//   OmniSketch : Punter, Papapetrou, Garofalakis (PVLDB 17(3), 2023), re-implemented
//   USS        : Unbiased Space-Saving over full attribute tuples (Ting, SIGMOD 2018)
//   CoStrata   : coordinated, hash-stratified row sample with count-driven thresholds (this work)
#pragma once
#include "common.hpp"

// =============================================================== Reservoir
struct Reservoir {
    uint32_t p;
    uint64_t M, N = 0, size = 0, next;
    double W;
    Rng rng;
    std::vector<std::vector<uint32_t>> col;

    Reservoir(uint32_t p_, uint64_t M_, uint64_t seed) : p(p_), M(M_), rng(seed), col(p_) {
        for (auto& c : col) c.resize(M);
        W = std::exp(std::log(rng.unit() + 1e-300) / (double)M);
        next = M + skip();
    }
    uint64_t skip() { return (uint64_t)std::floor(std::log(rng.unit() + 1e-300) / std::log1p(-W)) + 1; }
    static uint64_t bits_per_row(uint32_t p) { return 32ull * p; }
    uint64_t mem_bits() const { return M * bits_per_row(p); }

    inline void insert(uint64_t /*rid*/, const uint32_t* rec) {
        uint64_t i = N++;
        if (i < M) {
            for (uint32_t a = 0; a < p; a++) col[a][i] = rec[a];
            size = i + 1;
        } else if (i + 1 == next) {
            uint64_t s = rng.below(M);
            for (uint32_t a = 0; a < p; a++) col[a][s] = rec[a];
            W *= std::exp(std::log(rng.unit() + 1e-300) / (double)M);
            next += skip();
        }
    }
    void finalize() {}
    double estimate(const Query& q) const {
        if (size == 0) return 0;
        const uint32_t* c0 = col[q.attrs[0]].data();
        uint32_t v0 = q.vals[0];
        uint64_t hits = 0;
        for (uint64_t s = 0; s < size; s++) {
            if (c0[s] != v0) continue;
            bool ok = true;
            for (size_t k = 1; k < q.attrs.size(); k++)
                if (col[q.attrs[k]][s] != q.vals[k]) { ok = false; break; }
            hits += ok;
        }
        return (double)hits * (double)N / (double)size;
    }
};

// =============================================================== OmniSketch
// p attribute sketches, each d x w cells; a cell keeps a counter and the B
// smallest hashes g(rid) of the records hashed into it (bottom-B / K-minwise).
// Estimator (Eq. 4 of the paper): n_max / B * | intersection of the p_q * d samples |.
// Engineering beyond the reference Java code: bounded binary max-heaps in flat
// vectors instead of red-black trees, and a dense threshold array so that the
// common "reject" path costs one comparison per cell.
struct OmniSketch {
    uint32_t p, d, w;
    uint64_t B;
    uint64_t N = 0, gseed;
    std::vector<uint64_t> hseed;             // p*d
    std::vector<uint32_t> cnt;               // p*d*w
    std::vector<uint64_t> thr;               // p*d*w : current max hash once full, else UINT64_MAX
    std::vector<std::vector<uint64_t>> smp;  // p*d*w : heap during ingest, sorted after finalize()
    bool sorted = false;

    OmniSketch(uint32_t p_, uint32_t d_, uint32_t w_, uint64_t B_, uint64_t seed)
        : p(p_), d(d_), w(w_), B(B_), hseed((size_t)p_ * d_), cnt((size_t)p_ * d_ * w_, 0),
          thr((size_t)p_ * d_ * w_, UINT64_MAX), smp((size_t)p_ * d_ * w_) {
        Rng r(seed);
        gseed = r.next();
        for (auto& s : hseed) s = r.next();
    }
    inline size_t cell(uint32_t a, uint32_t j, uint32_t v) const {
        return ((size_t)a * d + j) * w + bucket_of(v, hseed[(size_t)a * d + j], w);
    }
    // "Lean" accounting: 32 bits per stored hash + one 32-bit counter per cell.
    uint64_t mem_bits_lean() const {
        uint64_t e = 0;
        for (auto& s : smp) e += s.size();
        return 32ull * e + 32ull * cnt.size();
    }
    // Accounting used in the paper / reference code: B*(b + 3*32 + 1) + 32 bits per cell.
    static uint64_t paper_bits(uint32_t p, uint32_t d, uint32_t w, uint64_t B, double delta = 0.1) {
        double b = std::ceil(std::log2(4.0 * std::pow((double)B, 2.5) / delta));
        return (uint64_t)((double)p * d * w * ((double)B * (b + 97.0) + 32.0));
    }
    // Parameters prescribed by the paper for (eps, delta).
    static void paper_wd(double eps, double delta, uint32_t& w, uint32_t& d) {
        d = (uint32_t)std::ceil(std::log(2.0 / delta));
        w = 1 + (uint32_t)std::ceil(std::exp(1.0) * std::pow((1.0 + eps) / eps, 1.0 / d));
    }

    inline void insert(uint64_t rid, const uint32_t* rec) {
        N++;
        uint64_t g = mix64(rid ^ gseed) >> 1;
        for (uint32_t a = 0; a < p; a++) {
            for (uint32_t j = 0; j < d; j++) {
                size_t c = cell(a, j, rec[a]);
                cnt[c]++;
                if (g >= thr[c]) continue;
                auto& h = smp[c];
                if (h.size() < B) {
                    h.push_back(g);
                    std::push_heap(h.begin(), h.end());
                    if (h.size() == B) thr[c] = h.front();
                } else {
                    std::pop_heap(h.begin(), h.end());
                    h.back() = g;
                    std::push_heap(h.begin(), h.end());
                    thr[c] = h.front();
                }
            }
        }
    }
    void finalize() {
        for (auto& h : smp) std::sort(h.begin(), h.end());
        sorted = true;
    }
    double estimate(const Query& q) const {
        std::vector<size_t> cells;
        uint32_t nmax = 0;
        for (size_t k = 0; k < q.attrs.size(); k++)
            for (uint32_t j = 0; j < d; j++) {
                size_t c = cell(q.attrs[k], j, q.vals[k]);
                cells.push_back(c);
                nmax = std::max(nmax, cnt[c]);
            }
        std::sort(cells.begin(), cells.end(), [&](size_t x, size_t y) { return smp[x].size() < smp[y].size(); });
        std::vector<uint64_t> cand = smp[cells[0]], tmp;
        for (size_t i = 1; i < cells.size() && !cand.empty(); i++) {
            const auto& s = smp[cells[i]];
            tmp.clear();
            std::set_intersection(cand.begin(), cand.end(), s.begin(), s.end(), std::back_inserter(tmp));
            cand.swap(tmp);
        }
        double scale = std::max(1.0, (double)nmax / (double)B);
        return scale * (double)cand.size();
    }
};

// =============================================================== USS
// Unbiased Space-Saving on the full p-attribute tuple. Subset sums (any
// predicate over the tuple) are estimated by summing the counters of the
// matching monitored tuples.
struct USS {
    uint32_t p;
    uint64_t m, size = 0, N = 0, tmask;
    Rng rng;
    std::vector<uint32_t> keys, cnt, heap, hpos;
    std::vector<int32_t> table;
    uint64_t hseed;

    USS(uint32_t p_, uint64_t m_, uint64_t seed) : p(p_), m(m_), rng(seed) {
        keys.resize(m * p);
        cnt.assign(m, 0);
        heap.resize(m);
        hpos.resize(m);
        uint64_t cap = 16;
        while (cap < 2 * m) cap <<= 1;
        table.assign(cap, -1);
        tmask = cap - 1;
        hseed = rng.next();
    }
    static uint64_t bits_per_row(uint32_t p) { return 32ull * p + 64; }
    uint64_t mem_bits() const { return m * bits_per_row(p); }
    inline uint64_t hkey(const uint32_t* r) const {
        uint64_t h = hseed;
        for (uint32_t a = 0; a < p; a++) h = (h ^ r[a]) * 0x9E3779B97F4A7C15ULL, h ^= h >> 29;
        return mix64(h);
    }
    inline bool eq(uint32_t s, const uint32_t* r) const { return memcmp(&keys[(size_t)s * p], r, 4 * p) == 0; }
    void sift_down(uint64_t i) {
        uint32_t s = heap[i];
        uint32_t c = cnt[s];
        for (;;) {
            uint64_t l = 2 * i + 1;
            if (l >= size) break;
            if (l + 1 < size && cnt[heap[l + 1]] < cnt[heap[l]]) l++;
            if (cnt[heap[l]] >= c) break;
            heap[i] = heap[l];
            hpos[heap[i]] = (uint32_t)i;
            i = l;
        }
        heap[i] = s;
        hpos[s] = (uint32_t)i;
    }
    void sift_up(uint64_t i) {
        uint32_t s = heap[i];
        uint32_t c = cnt[s];
        while (i > 0) {
            uint64_t par = (i - 1) / 2;
            if (cnt[heap[par]] <= c) break;
            heap[i] = heap[par];
            hpos[heap[i]] = (uint32_t)i;
            i = par;
        }
        heap[i] = s;
        hpos[s] = (uint32_t)i;
    }
    void tab_insert(uint32_t s) {
        uint64_t i = hkey(&keys[(size_t)s * p]) & tmask;
        while (table[i] >= 0) i = (i + 1) & tmask;
        table[i] = (int32_t)s;
    }
    void tab_erase(uint32_t s) {
        uint64_t i = hkey(&keys[(size_t)s * p]) & tmask;
        while (table[i] != (int32_t)s) i = (i + 1) & tmask;
        // backward-shift deletion for linear probing
        uint64_t j = i;
        for (;;) {
            j = (j + 1) & tmask;
            if (table[j] < 0) break;
            uint64_t k = hkey(&keys[(size_t)table[j] * p]) & tmask;
            bool movable = (i <= j) ? (k <= i || k > j) : (k <= i && k > j);
            if (movable) {
                table[i] = table[j];
                i = j;
            }
        }
        table[i] = -1;
    }
    inline void insert(uint64_t /*rid*/, const uint32_t* rec) {
        N++;
        uint64_t i = hkey(rec) & tmask;
        while (table[i] >= 0) {
            if (eq((uint32_t)table[i], rec)) {
                uint32_t s = (uint32_t)table[i];
                cnt[s]++;
                sift_down(hpos[s]);
                return;
            }
            i = (i + 1) & tmask;
        }
        if (size < m) {
            uint32_t s = (uint32_t)size++;
            memcpy(&keys[(size_t)s * p], rec, 4 * p);
            cnt[s] = 1;
            heap[s] = s;
            hpos[s] = s;
            sift_up(s);
            table[i] = (int32_t)s;
            return;
        }
        uint32_t s = heap[0];
        cnt[s]++;
        if (rng.unit() * (double)cnt[s] < 1.0) {
            tab_erase(s);
            memcpy(&keys[(size_t)s * p], rec, 4 * p);
            tab_insert(s);
        }
        sift_down(0);
    }
    void finalize() {}
    double estimate(const Query& q) const {
        uint64_t sum = 0;
        for (uint64_t s = 0; s < size; s++)
            if (q.matches(&keys[s * p])) sum += cnt[s];
        return (double)sum;
    }
};

// =============================================================== CoStrata
// One Count-Min-style row of w exact counters per attribute defines p hash
// stratifications of the stream (plus the trivial stratification {all}).
// A record r with uniform priority u_r in [0,1) is retained iff
//        u_r * nstar_r < Bcur ,   nstar_r = min( G , min_a s(n_{a,h_a(r_a)}) ),
// i.e. it is sampled with probability pi_r = min(1, Bcur / nstar_r): the
// *maximum* over the sampling rates of all strata it belongs to. Thresholds
// are functions of counters only (no per-stratum heaps); the single global
// parameter Bcur is lowered by selection whenever the row store is full.
struct CoStrata {
    uint32_t p, w;
    uint64_t M;          // row capacity
    double lf;           // weight of the global ("house") stratum relative to all hash strata; 0 disables
    double gamma;        // allocation exponent: s(n) = n^gamma (1 = equal allocation per stratum)
    double slack;        // fraction of the store freed by a compaction
    uint64_t N = 0, size = 0, E = 0, compactions = 0;
    double Bcur = std::numeric_limits<double>::infinity();
    double G = 0;        // running max of N / (lf * E)
    uint64_t useed;
    std::vector<uint64_t> hseed;                // p
    std::vector<uint32_t> cnt;                  // p*w
    std::vector<std::vector<uint32_t>> col;     // p columns of M
    std::vector<double> u;                      // M priorities
    // query-time working state (built by finalize)
    std::vector<double> invpi, Wb, Vb;

    CoStrata(uint32_t p_, uint32_t w_, uint64_t M_, double lf_, double gamma_, uint64_t seed, double slack_ = 0.1)
        : p(p_), w(w_), M(M_), lf(lf_), gamma(gamma_), slack(slack_), hseed(p_), cnt((size_t)p_ * w_, 0), col(p_), u(M_) {
        Rng r(seed);
        useed = r.next();
        for (auto& s : hseed) s = r.next();
        for (auto& c : col) c.resize(M);
    }
    static uint64_t bits_per_row(uint32_t p) { return 32ull * p + 64; }
    uint64_t mem_bits() const { return M * bits_per_row(p) + 32ull * cnt.size(); }
    static uint64_t rows_for_budget(uint64_t bits, uint32_t p, uint32_t w) {
        uint64_t fixed = 32ull * p * w;
        return bits > fixed ? (bits - fixed) / bits_per_row(p) : 0;
    }
    inline double sfun(double n) const { return gamma == 1.0 ? n : std::pow(n, gamma); }
    inline double nstar_slot(uint64_t s) const {
        uint32_t nmin = UINT32_MAX;
        for (uint32_t a = 0; a < p; a++) nmin = std::min(nmin, cnt[(size_t)a * w + bucket_of(col[a][s], hseed[a], w)]);
        double v = sfun((double)nmin);
        return (lf > 0 && G < v) ? G : v;
    }
    inline void insert(uint64_t rid, const uint32_t* rec) {
        N++;
        uint32_t nmin = UINT32_MAX;
        for (uint32_t a = 0; a < p; a++) {
            uint32_t& c = cnt[(size_t)a * w + bucket_of(rec[a], hseed[a], w)];
            if (c == 0) E++;
            c++;
            nmin = std::min(nmin, c);
        }
        double ns = sfun((double)nmin);
        if (lf > 0) {
            double g = (double)N / (lf * (double)E);
            if (g > G) G = g;
            if (G < ns) ns = G;
        }
        double ur = to_unit(mix64(rid ^ useed));
        if (ur * ns < Bcur) {
            uint64_t s = size++;
            for (uint32_t a = 0; a < p; a++) col[a][s] = rec[a];
            u[s] = ur;
            if (size == M) compact();
        }
    }
    void compact() {
        compactions++;
        std::vector<double> z(size), zs;
        for (uint64_t s = 0; s < size; s++) z[s] = u[s] * nstar_slot(s);
        uint64_t K = (uint64_t)((1.0 - slack) * (double)M);
        if (K >= size) K = size - 1;
        zs = z;
        std::nth_element(zs.begin(), zs.begin() + K, zs.end());
        if (zs[K] < Bcur) Bcur = zs[K];
        uint64_t o = 0;
        for (uint64_t s = 0; s < size; s++) {
            if (z[s] < Bcur) {
                if (o != s) {
                    for (uint32_t a = 0; a < p; a++) col[a][o] = col[a][s];
                    u[o] = u[s];
                }
                o++;
            }
        }
        size = o;
    }
    // Drop stale rows, compute Horvitz-Thompson weights and per-stratum aggregates.
    void finalize() {
        uint64_t o = 0;
        invpi.assign(size, 0.0);
        Wb.assign((size_t)p * w, 0.0);
        Vb.assign((size_t)p * w, 0.0);
        for (uint64_t s = 0; s < size; s++) {
            double ns = nstar_slot(s);
            if (!(u[s] * ns < Bcur)) continue;
            double pi = std::min(1.0, Bcur / ns);
            if (o != s) {
                for (uint32_t a = 0; a < p; a++) col[a][o] = col[a][s];
                u[o] = u[s];
            }
            invpi[o] = 1.0 / pi;
            for (uint32_t a = 0; a < p; a++) {
                size_t b = (size_t)a * w + bucket_of(col[a][o], hseed[a], w);
                Wb[b] += 1.0 / pi;
                Vb[b] += (1.0 - pi) / (pi * pi);
            }
            o++;
        }
        size = o;
        invpi.resize(size);
    }
    struct Est {
        double hajek, ht, sd_hajek, sd_ht, cm; // cm = Count-Min style upper bound from the counters
    };
    Est estimate(const Query& q) const {
        const uint32_t* c0 = col[q.attrs[0]].data();
        uint32_t v0 = q.vals[0];
        double num = 0, A = 0;
        for (uint64_t s = 0; s < size; s++) {
            if (c0[s] != v0) continue;
            bool ok = true;
            for (size_t k = 1; k < q.attrs.size(); k++)
                if (col[q.attrs[k]][s] != q.vals[k]) { ok = false; break; }
            if (ok) {
                double ip = invpi[s];
                num += ip;
                A += ip * ip - ip; // (1-pi)/pi^2
            }
        }
        // lightest stratum touched by the query
        size_t best = 0;
        uint32_t nb = UINT32_MAX;
        for (size_t k = 0; k < q.attrs.size(); k++) {
            size_t b = (size_t)q.attrs[k] * w + bucket_of(q.vals[k], hseed[q.attrs[k]], w);
            if (cnt[b] < nb) nb = cnt[b], best = b;
        }
        Est e;
        e.ht = num;
        e.sd_ht = std::sqrt(A);
        e.cm = (double)nb;
        if (Wb[best] > 0) {
            double R = num / Wb[best];
            double scale = (double)nb / Wb[best];
            e.hajek = (double)nb * R;
            double var = scale * scale * ((1 - R) * (1 - R) * A + R * R * std::max(0.0, Vb[best] - A));
            e.sd_hajek = std::sqrt(std::max(0.0, var));
        } else {
            e.hajek = 0;
            e.sd_hajek = 0;
        }
        return e;
    }
};

// =============================================================== USS + calibration
// USS over full tuples plus one row of w exact counters per attribute; the
// subset-sum estimate is ratio-calibrated to the exact count of the lightest
// hash stratum touched by the query (same calibration as CoStrata).
struct USSC {
    USS uss;
    uint32_t p, w;
    std::vector<uint64_t> hseed;
    std::vector<uint32_t> cnt;
    std::vector<double> Wb;
    USSC(uint32_t p_, uint32_t w_, uint64_t m, uint64_t seed) : uss(p_, m, seed), p(p_), w(w_), hseed(p_), cnt((size_t)p_ * w_, 0) {
        Rng r(seed ^ 0x5bd1e995);
        for (auto& x : hseed) x = r.next();
    }
    static uint64_t rows_for_budget(uint64_t bits, uint32_t p, uint32_t w) {
        uint64_t fixed = 32ull * p * w;
        return bits > fixed ? (bits - fixed) / USS::bits_per_row(p) : 0;
    }
    uint64_t mem_bits() const { return uss.mem_bits() + 32ull * cnt.size(); }
    inline void insert(uint64_t rid, const uint32_t* rec) {
        for (uint32_t a = 0; a < p; a++) cnt[(size_t)a * w + bucket_of(rec[a], hseed[a], w)]++;
        uss.insert(rid, rec);
    }
    void finalize() {
        Wb.assign((size_t)p * w, 0.0);
        for (uint64_t s = 0; s < uss.size; s++)
            for (uint32_t a = 0; a < p; a++)
                Wb[(size_t)a * w + bucket_of(uss.keys[s * p + a], hseed[a], w)] += uss.cnt[s];
    }
    double estimate(const Query& q) const {
        double num = uss.estimate(q);
        size_t best = 0;
        uint32_t nb = UINT32_MAX;
        for (size_t k = 0; k < q.attrs.size(); k++) {
            size_t b = (size_t)q.attrs[k] * w + bucket_of(q.vals[k], hseed[q.attrs[k]], w);
            if (cnt[b] < nb) nb = cnt[b], best = b;
        }
        return Wb[best] > 0 ? (double)nb * num / Wb[best] : 0.0;
    }
};


In [ ]:
%%writefile src/bench.cpp
// Benchmark driver.
//   bench acc  <data.bin> <workload.txt> <out.csv> <name> <mem_mb> <seed> [-p k] <spec>...
//   bench time <data.bin> <name> <mem_mb> <seed> <reps> [-p k] <spec>...
// Method specs (all sized to the same memory budget of mem_mb * 8e6 bits):
//   res            uniform row reservoir
//   uss            Unbiased Space-Saving on full tuples
//   omnip          OmniSketch, paper parameters (eps=delta=0.1) and paper memory accounting
//   omnil          OmniSketch, paper (w,d), lean accounting (32 bits/sample, dense)
//   omni:w:d       OmniSketch with given (w,d); B chosen by an oracle so that the
//                  *actually stored* samples fill the budget (most favourable accounting)
//   cs:w:lf:gamma  CoStrata with w buckets/attribute, house weight lf, exponent gamma
// "acc" appends one line per (method, query) to out.csv and prints a RUN line.
#include "synopses.hpp"

struct Spec {
    std::string raw, kind;
    std::vector<double> a;
};
static Spec parse_spec(const std::string& s) {
    Spec sp;
    sp.raw = s;
    size_t pos = s.find(':');
    sp.kind = s.substr(0, pos);
    while (pos != std::string::npos) {
        size_t nx = s.find(':', pos + 1);
        sp.a.push_back(atof(s.substr(pos + 1, nx == std::string::npos ? nx : nx - pos - 1).c_str()));
        pos = nx;
    }
    return sp;
}

template <class S>
static double ingest(S& syn, const Dataset& d) {
    double t0 = now_sec();
    for (uint64_t i = 0; i < d.n; i++) syn.insert(i, d.row(i));
    return now_sec() - t0;
}

// w = 0 selects the default: the largest power of two such that the counters use <= 5% of the budget.
static uint32_t auto_w(double wspec, uint64_t bits, uint32_t p) {
    if (wspec > 0) return (uint32_t)wspec;
    uint32_t w = 64;
    while (w < 65536 && 32.0 * p * (2.0 * w) <= 0.05 * (double)bits) w *= 2;
    return w;
}

static OmniSketch* make_omni(const Spec& sp, const Dataset& d, uint64_t bits, uint64_t seed, std::string& info) {
    uint32_t w, dd;
    uint64_t B = 1;
    if (sp.kind == "omnip" || sp.kind == "omnil") {
        OmniSketch::paper_wd(0.1, 0.1, w, dd);
        uint64_t lo = 1, hi = 1ull << 34;
        while (lo < hi) {
            uint64_t mid = (lo + hi + 1) / 2;
            uint64_t need = sp.kind == "omnip" ? OmniSketch::paper_bits(d.p, dd, w, mid)
                                               : (uint64_t)d.p * dd * w * (32 * mid + 32);
            if (need <= bits) lo = mid; else hi = mid - 1;
        }
        B = lo;
    } else {
        w = (uint32_t)sp.a[0];
        dd = (uint32_t)sp.a[1];
        OmniSketch probe(d.p, dd, w, 1, seed);
        std::vector<uint32_t> c((size_t)d.p * dd * w, 0);
        for (uint64_t i = 0; i < d.n; i++) {
            const uint32_t* r = d.row(i);
            for (uint32_t a = 0; a < d.p; a++)
                for (uint32_t j = 0; j < dd; j++) c[probe.cell(a, j, r[a])]++;
        }
        uint64_t fixed = 32ull * c.size();
        uint64_t budget = bits > fixed ? (bits - fixed) / 32 : 0;
        uint64_t lo = 1, hi = d.n;
        while (lo < hi) {
            uint64_t mid = (lo + hi + 1) / 2, tot = 0;
            for (uint32_t x : c) tot += std::min<uint64_t>(x, mid);
            if (tot <= budget) lo = mid; else hi = mid - 1;
        }
        B = lo;
    }
    char buf[128];
    snprintf(buf, sizeof buf, "w=%u;d=%u;B=%llu", w, dd, (unsigned long long)B);
    info = buf;
    return new OmniSketch(d.p, dd, w, B, seed);
}

int main(int argc, char** argv) {
    if (argc < 2) return 1;
    std::string mode = argv[1];
    Dataset d;
    if (mode == "acc") {
        if (argc < 9) { fprintf(stderr, "bad args\n"); return 1; }
        if (!d.load(argv[2])) { fprintf(stderr, "cannot load data\n"); return 1; }
        std::vector<Query> qs;
        if (!load_workload(argv[3], qs)) { fprintf(stderr, "cannot load workload\n"); return 1; }
        FILE* out = fopen(argv[4], "a");
        std::string name = argv[5];
        double mem_mb = atof(argv[6]);
        uint64_t seed = strtoull(argv[7], 0, 10);
        uint64_t bits = (uint64_t)(mem_mb * 8e6);
        int ai = 8;
        if (std::string(argv[ai]) == "-p") { d.project_prefix((uint32_t)atoi(argv[ai + 1])); ai += 2; }
        for (; ai < argc; ai++) {
            Spec sp = parse_spec(argv[ai]);
            std::string info;
            double t_ing = 0, t_q = 0;
            uint64_t used_bits = 0;
            std::vector<double> est(qs.size()), ht, sd, cm;
            if (sp.kind == "res") {
                Reservoir s(d.p, std::max<uint64_t>(1, bits / Reservoir::bits_per_row(d.p)), seed * 7919 + 1);
                t_ing = ingest(s, d);
                s.finalize();
                double t0 = now_sec();
                for (size_t i = 0; i < qs.size(); i++) est[i] = s.estimate(qs[i]);
                t_q = now_sec() - t0;
                used_bits = s.mem_bits();
                info = "M=" + std::to_string(s.M);
            } else if (sp.kind == "uss") {
                USS s(d.p, std::max<uint64_t>(1, bits / USS::bits_per_row(d.p)), seed * 7919 + 2);
                t_ing = ingest(s, d);
                double t0 = now_sec();
                for (size_t i = 0; i < qs.size(); i++) est[i] = s.estimate(qs[i]);
                t_q = now_sec() - t0;
                used_bits = s.mem_bits();
                info = "m=" + std::to_string(s.m) + ";held=" + std::to_string(s.size);
            } else if (sp.kind == "ussc") {
                uint32_t w = auto_w(sp.a.empty() ? 0 : sp.a[0], bits, d.p);
                USSC s(d.p, w, std::max<uint64_t>(1, USSC::rows_for_budget(bits, d.p, w)), seed * 7919 + 2);
                t_ing = ingest(s, d);
                s.finalize();
                double t0 = now_sec();
                for (size_t i = 0; i < qs.size(); i++) est[i] = s.estimate(qs[i]);
                t_q = now_sec() - t0;
                used_bits = s.mem_bits();
                info = "m=" + std::to_string(s.uss.m) + ";w=" + std::to_string(w);
            } else if (sp.kind.rfind("omni", 0) == 0) {
                OmniSketch* s = make_omni(sp, d, bits, seed * 7919 + 3, info);
                t_ing = ingest(*s, d);
                s->finalize();
                double t0 = now_sec();
                for (size_t i = 0; i < qs.size(); i++) est[i] = s->estimate(qs[i]);
                t_q = now_sec() - t0;
                used_bits = sp.kind == "omnip" ? OmniSketch::paper_bits(s->p, s->d, s->w, s->B) : s->mem_bits_lean();
                delete s;
            } else if (sp.kind == "cs") {
                uint32_t w = auto_w(sp.a.empty() ? 0 : sp.a[0], bits, d.p);
                double lf = sp.a.size() > 1 ? sp.a[1] : 1.0, gamma = sp.a.size() > 2 ? sp.a[2] : 1.0;
                uint64_t M = CoStrata::rows_for_budget(bits, d.p, w);
                if (M < 16) { fprintf(stderr, "budget too small for %s\n", sp.raw.c_str()); continue; }
                CoStrata s(d.p, w, M, lf, gamma, seed * 7919 + 4);
                t_ing = ingest(s, d);
                s.finalize();
                ht.resize(qs.size()); sd.resize(qs.size()); cm.resize(qs.size());
                double t0 = now_sec();
                for (size_t i = 0; i < qs.size(); i++) {
                    CoStrata::Est e = s.estimate(qs[i]);
                    est[i] = e.hajek; ht[i] = e.ht; sd[i] = e.sd_hajek; cm[i] = e.cm;
                }
                t_q = now_sec() - t0;
                used_bits = s.mem_bits();
                char buf[160];
                snprintf(buf, sizeof buf, "w=%u;M=%llu;held=%llu;B=%.4g;compactions=%llu;E=%llu", w, (unsigned long long)M,
                         (unsigned long long)s.size, s.Bcur, (unsigned long long)s.compactions, (unsigned long long)s.E);
                info = buf;
            } else {
                fprintf(stderr, "unknown spec %s\n", sp.raw.c_str());
                continue;
            }
            for (size_t i = 0; i < qs.size(); i++) {
                if (ht.empty()) fprintf(out, "%s,%s,%g,%llu,%zu,%.6g,,,\n", name.c_str(), sp.raw.c_str(), mem_mb, (unsigned long long)seed, i, est[i]);
                else fprintf(out, "%s,%s,%g,%llu,%zu,%.6g,%.6g,%.6g,%.6g\n", name.c_str(), sp.raw.c_str(), mem_mb, (unsigned long long)seed, i, est[i], ht[i], sd[i], cm[i]);
            }
            printf("RUN,%s,%s,%g,%llu,%u,%llu,%.3f,%.2f,%.2f,%.3f,%s\n", name.c_str(), sp.raw.c_str(), mem_mb,
                   (unsigned long long)seed, d.p, (unsigned long long)d.n, t_ing, t_ing / (double)d.n * 1e9,
                   t_q / (double)qs.size() * 1e6, (double)used_bits / 8e6, info.c_str());
            fflush(stdout);
        }
        fclose(out);
    } else if (mode == "time") {
        if (argc < 8) { fprintf(stderr, "bad args\n"); return 1; }
        if (!d.load(argv[2])) { fprintf(stderr, "cannot load data\n"); return 1; }
        std::string name = argv[3];
        double mem_mb = atof(argv[4]);
        uint64_t seed = strtoull(argv[5], 0, 10);
        int reps = atoi(argv[6]);
        uint64_t bits = (uint64_t)(mem_mb * 8e6);
        int ai = 7;
        if (std::string(argv[ai]) == "-p") { d.project_prefix((uint32_t)atoi(argv[ai + 1])); ai += 2; }
        for (; ai < argc; ai++) {
            Spec sp = parse_spec(argv[ai]);
            std::vector<double> ts;
            std::string info;
            for (int r = 0; r < reps; r++) {
                if (sp.kind == "res") {
                    Reservoir s(d.p, std::max<uint64_t>(1, bits / Reservoir::bits_per_row(d.p)), seed + r);
                    ts.push_back(ingest(s, d));
                } else if (sp.kind == "uss") {
                    USS s(d.p, std::max<uint64_t>(1, bits / USS::bits_per_row(d.p)), seed + r);
                    ts.push_back(ingest(s, d));
                } else if (sp.kind.rfind("omni", 0) == 0) {
                    OmniSketch* s = make_omni(sp, d, bits, seed + r, info);
                    ts.push_back(ingest(*s, d));
                    delete s;
                } else if (sp.kind == "ussc") {
                    uint32_t w = auto_w(sp.a.empty() ? 0 : sp.a[0], bits, d.p);
                    USSC s(d.p, w, std::max<uint64_t>(1, USSC::rows_for_budget(bits, d.p, w)), seed + r);
                    ts.push_back(ingest(s, d));
                } else if (sp.kind == "cs") {
                    uint32_t w = auto_w(sp.a.empty() ? 0 : sp.a[0], bits, d.p);
                    double lf = sp.a.size() > 1 ? sp.a[1] : 1.0, gamma = sp.a.size() > 2 ? sp.a[2] : 1.0;
                    CoStrata s(d.p, w, CoStrata::rows_for_budget(bits, d.p, w), lf, gamma, seed + r);
                    ts.push_back(ingest(s, d));
                    info = "compactions=" + std::to_string(s.compactions);
                }
            }
            std::sort(ts.begin(), ts.end());
            double med = ts[ts.size() / 2];
            printf("TIME,%s,%s,%g,%u,%llu,%.2f,%.3f,%s\n", name.c_str(), sp.raw.c_str(), mem_mb, d.p,
                   (unsigned long long)d.n, med / (double)d.n * 1e9, (double)d.n / med / 1e6, info.c_str());
            fflush(stdout);
        }
    }
    return 0;
}


In [ ]:
%%writefile src/gen_synth.cpp
// Synthetic stream generator.
//   gen_synth <out.bin> <n> <p> <alpha> <domain> <seed>
// Every attribute is drawn independently from Zipf(alpha) over {1..domain}
// (alpha = 0 gives the uniform distribution). This mirrors the synthetic
// streams of the OmniSketch paper (p = 5, domain = 10000, n = 5M).
#include "common.hpp"

int main(int argc, char** argv) {
    if (argc < 7) {
        fprintf(stderr, "usage: gen_synth out n p alpha domain seed\n");
        return 1;
    }
    Dataset d;
    d.n = strtoull(argv[2], 0, 10);
    d.p = (uint32_t)atoi(argv[3]);
    double alpha = atof(argv[4]);
    uint32_t dom = (uint32_t)atoi(argv[5]);
    Rng rng(strtoull(argv[6], 0, 10));
    std::vector<double> cdf(dom);
    double s = 0;
    for (uint32_t k = 1; k <= dom; k++) s += std::pow((double)k, -alpha), cdf[k - 1] = s;
    for (auto& c : cdf) c /= s;
    // Per-attribute random relabeling so that value ids carry no rank information.
    std::vector<std::vector<uint32_t>> perm(d.p, std::vector<uint32_t>(dom));
    for (uint32_t a = 0; a < d.p; a++) {
        for (uint32_t k = 0; k < dom; k++) perm[a][k] = k + 1;
        for (uint32_t k = dom - 1; k > 0; k--) std::swap(perm[a][k], perm[a][rng.below(k + 1)]);
    }
    d.v.resize(d.n * d.p);
    for (uint64_t i = 0; i < d.n * d.p; i++) {
        double x = rng.unit();
        uint32_t k = (uint32_t)(std::lower_bound(cdf.begin(), cdf.end(), x) - cdf.begin());
        if (k >= dom) k = dom - 1;
        d.v[i] = perm[i % d.p][k];
    }
    if (!d.save(argv[1])) {
        fprintf(stderr, "cannot write %s\n", argv[1]);
        return 1;
    }
    fprintf(stderr, "wrote %s: n=%llu p=%u alpha=%.2f domain=%u\n", argv[1], (unsigned long long)d.n, d.p, alpha, dom);
    return 0;
}


In [ ]:
%%writefile src/mkworkload.cpp
// Workload generator with exact ground truth.
//   mkworkload <data.bin> <out.txt> <queries_per_k> <kmax> <seed>
// Two families of conjunctive equality queries (all non-empty by construction):
//   kind 0  "record-sampled": pick a record uniformly at random, pick k random
//           attributes, use the record's values. This is the workload of the
//           OmniSketch paper; it favours frequent values.
//   kind 1  "frequency-stratified anchor": pick an attribute, pick a frequency
//           decade [10^j, 10^(j+1)) uniformly among the non-empty ones (j >= 1),
//           pick a value of that attribute uniformly inside the decade, pick a
//           random record carrying it, and add k-1 further predicates taken
//           from that record. This covers rare and frequent anchors evenly
//           (the "drill-down on one entity" query shape).
#include "common.hpp"
#include <unordered_map>
#include <unordered_set>

struct VInfo {
    uint32_t count = 0;
    uint32_t rec = 0;
};

int main(int argc, char** argv) {
    if (argc < 6) {
        fprintf(stderr, "usage: mkworkload data out queries_per_k kmax seed [prefix_p]\n");
        return 1;
    }
    Dataset d;
    if (!d.load(argv[1])) {
        fprintf(stderr, "cannot load %s\n", argv[1]);
        return 1;
    }
    if (argc > 6) d.project_prefix((uint32_t)atoi(argv[6]));
    int per_k = atoi(argv[3]);
    uint32_t kmax = std::min<uint32_t>((uint32_t)atoi(argv[4]), d.p);
    Rng rng(strtoull(argv[5], 0, 10));
    const uint32_t p = d.p;

    // per-attribute value statistics with a size-1 reservoir of carrying records
    std::vector<std::unordered_map<uint32_t, VInfo>> stats(p);
    for (uint64_t i = 0; i < d.n; i++) {
        const uint32_t* r = d.row(i);
        for (uint32_t a = 0; a < p; a++) {
            VInfo& vi = stats[a][r[a]];
            vi.count++;
            if (rng.below(vi.count) == 0) vi.rec = (uint32_t)i;
        }
    }
    // decade lists
    std::vector<std::vector<std::vector<uint32_t>>> dec(p); // dec[a][j] = values with count in [10^j,10^(j+1))
    std::vector<uint32_t> eligible;
    for (uint32_t a = 0; a < p; a++) {
        dec[a].resize(10);
        for (auto& kv : stats[a]) {
            int j = (int)std::floor(std::log10((double)kv.second.count));
            dec[a][j].push_back(kv.first);
        }
        for (auto& l : dec[a]) std::sort(l.begin(), l.end());
        fprintf(stderr, "attr %u: %zu distinct\n", a, stats[a].size());
        if (stats[a].size() >= 20) eligible.push_back(a);
    }

    std::vector<Query> qs;
    std::unordered_set<std::string> seen;
    auto add_query = [&](int kind, uint64_t rec, uint32_t k, int anchor) {
        std::vector<uint32_t> attrs;
        if (anchor >= 0) attrs.push_back((uint32_t)anchor);
        while (attrs.size() < k) {
            uint32_t a = (uint32_t)rng.below(p);
            if (std::find(attrs.begin(), attrs.end(), a) == attrs.end()) attrs.push_back(a);
        }
        std::vector<uint32_t> sorted = attrs;
        std::sort(sorted.begin(), sorted.end());
        std::string key;
        const uint32_t* r = d.row(rec);
        for (uint32_t a : sorted) key += std::to_string(a) + "=" + std::to_string(r[a]) + ";";
        if (!seen.insert(std::to_string(kind) + "|" + key).second) return false;
        Query q;
        q.kind = kind;
        for (uint32_t a : attrs) q.attrs.push_back(a), q.vals.push_back(r[a]);
        qs.push_back(std::move(q));
        return true;
    };
    for (uint32_t k = 1; k <= kmax; k++) {
        int made = 0, tries = 0;
        while (made < per_k && tries++ < per_k * 50) made += add_query(0, rng.below(d.n), k, -1);
        if (eligible.empty()) continue;
        made = 0, tries = 0;
        while (made < per_k && tries++ < per_k * 50) {
            uint32_t a = eligible[rng.below(eligible.size())];
            std::vector<int> js;
            for (int j = 1; j < 10; j++)
                if (!dec[a][j].empty()) js.push_back(j);
            if (js.empty()) continue;
            int j = js[rng.below(js.size())];
            uint32_t v = dec[a][j][rng.below(dec[a][j].size())];
            made += add_query(1, stats[a][v].rec, k, (int)a);
        }
    }
    // exact ground truth: index queries by their first predicate
    std::unordered_map<uint64_t, std::vector<uint32_t>> index;
    std::vector<char> used(p, 0);
    for (uint32_t i = 0; i < qs.size(); i++) {
        index[((uint64_t)qs[i].attrs[0] << 32) | qs[i].vals[0]].push_back(i);
        used[qs[i].attrs[0]] = 1;
    }
    for (uint64_t i = 0; i < d.n; i++) {
        const uint32_t* r = d.row(i);
        for (uint32_t a = 0; a < p; a++) {
            if (!used[a]) continue;
            auto it = index.find(((uint64_t)a << 32) | r[a]);
            if (it == index.end()) continue;
            for (uint32_t qi : it->second)
                if (qs[qi].matches(r)) qs[qi].truth++;
        }
    }
    FILE* f = fopen(argv[2], "w");
    for (auto& q : qs) {
        uint64_t mf = UINT64_MAX;
        for (size_t i = 0; i < q.attrs.size(); i++) mf = std::min<uint64_t>(mf, stats[q.attrs[i]][q.vals[i]].count);
        fprintf(f, "%d %zu %llu %llu", q.kind, q.attrs.size(), (unsigned long long)q.truth, (unsigned long long)mf);
        for (size_t i = 0; i < q.attrs.size(); i++) fprintf(f, " %u %u", q.attrs[i], q.vals[i]);
        fprintf(f, "\n");
    }
    fclose(f);
    // number of distinct full tuples (relevant for tuple-aggregating synopses)
    std::unordered_set<uint64_t> tup;
    for (uint64_t i = 0; i < d.n; i++) {
        const uint32_t* r = d.row(i);
        uint64_t h = 1469598103934665603ULL;
        for (uint32_t a = 0; a < p; a++) h = mix64(h ^ r[a]);
        tup.insert(h);
    }
    fprintf(stderr, "wrote %zu queries to %s; n=%llu p=%u distinct_tuples=%zu\n", qs.size(), argv[2],
            (unsigned long long)d.n, p, tup.size());
    return 0;
}


In [ ]:
%%writefile src/pcap2bin.cpp
// Convert a (gzipped) pcap trace into the binary stream format, one record per IPv4 packet.
//   pcap2bin <trace.pcap[.gz]> <out.bin> [max_packets]
// Attributes: srcIP, dstIP, srcPort, dstPort, proto, ipLen, src/16, dst/16, ttl, tcpFlags
#include "common.hpp"

static inline uint16_t be16(const unsigned char* b) { return (uint16_t)((b[0] << 8) | b[1]); }
static inline uint32_t be32(const unsigned char* b) { return ((uint32_t)b[0] << 24) | ((uint32_t)b[1] << 16) | ((uint32_t)b[2] << 8) | b[3]; }

int main(int argc, char** argv) {
    if (argc < 3) {
        fprintf(stderr, "usage: pcap2bin trace out [max]\n");
        return 1;
    }
    std::string path = argv[1];
    uint64_t maxp = argc > 3 ? strtoull(argv[3], 0, 10) : UINT64_MAX;
    bool gz = path.size() > 3 && path.substr(path.size() - 3) == ".gz";
    FILE* f = gz ? popen(("gzip -dc '" + path + "'").c_str(), "r") : fopen(path.c_str(), "rb");
    if (!f) return 1;
    unsigned char gh[24];
    if (fread(gh, 1, 24, f) != 24) return 1;
    uint32_t magic;
    memcpy(&magic, gh, 4);
    bool swap = (magic == 0xd4c3b2a1);
    if (magic != 0xa1b2c3d4 && !swap) {
        fprintf(stderr, "not a classic pcap file (magic %08x)\n", magic);
        return 1;
    }
    auto rd32 = [&](const unsigned char* b) {
        uint32_t x;
        memcpy(&x, b, 4);
        return swap ? __builtin_bswap32(x) : x;
    };
    uint32_t linktype = rd32(gh + 20);
    fprintf(stderr, "linktype=%u\n", linktype);
    Dataset d;
    d.p = 10;
    std::vector<unsigned char> buf(1 << 16);
    uint64_t total = 0, kept = 0;
    unsigned char ph[16];
    while (kept < maxp && fread(ph, 1, 16, f) == 16) {
        uint32_t incl = rd32(ph + 8);
        if (incl > buf.size()) buf.resize(incl);
        if (fread(buf.data(), 1, incl, f) != incl) break;
        total++;
        const unsigned char* b = buf.data();
        uint32_t off = 0;
        if (linktype == 1) { // Ethernet, possibly VLAN-tagged
            if (incl < 14) continue;
            uint16_t et = be16(b + 12);
            off = 14;
            while (et == 0x8100 && incl >= off + 4) et = be16(b + off + 2), off += 4;
            if (et != 0x0800) continue;
        } else if (linktype != 101 && linktype != 12) {
            continue;
        }
        if (incl < off + 20 || (b[off] >> 4) != 4) continue;
        const unsigned char* ip = b + off;
        uint32_t ihl = (ip[0] & 15u) * 4;
        uint32_t proto = ip[9], src = be32(ip + 12), dst = be32(ip + 16);
        uint32_t sport = 0, dport = 0, flags = 0;
        if ((proto == 6 || proto == 17) && incl >= off + ihl + 4) {
            sport = be16(ip + ihl);
            dport = be16(ip + ihl + 2);
            if (proto == 6 && incl >= off + ihl + 14) flags = ip[ihl + 13];
        }
        uint32_t rec[10] = {src, dst, sport, dport, proto, be16(ip + 2), src >> 16, dst >> 16, ip[8], flags};
        d.v.insert(d.v.end(), rec, rec + 10);
        kept++;
    }
    d.n = kept;
    d.save(argv[2]);
    fprintf(stderr, "packets=%llu ipv4_kept=%llu\n", (unsigned long long)total, (unsigned long long)kept);
    return 0;
}


In [ ]:
%%writefile scripts/run_accuracy.sh
#!/usr/bin/env bash
# Main accuracy matrix: every method at equal memory, 5 seeds.
# Usage: scripts/run_accuracy.sh [parallel_jobs]
set -u
cd "$(dirname "$0")/.."
J=${1:-8}
SKIP_LIST=${SKIP_LIST:-}
OUT=results/acc
mkdir -p "$OUT"
METHODS="res uss ussc omnip omnil omni:64:2 omni:256:2 omni:1024:2 omni:256:3 omni:1024:1 omni:4096:1 omni:4096:2 cs cs:0:1e9 cs:0:0"
jobs=()
for ds in zipf10_5M zipf13_5M zipf15_5M unif_5M taxi mawi; do
  for mem in 0.5 1 2 5 10 20 50; do
    for seed in 1 2 3 4 5; do
      jobs+=("$ds $mem $seed")
    done
  done
done
run_one() {
  ds=$1; mem=$2; seed=$3; shift 3
  f="$OUT/$ds.$mem.$seed"
  # jobs already finished on another machine (results are seed-deterministic) can be listed in $SKIP_LIST
  [ -n "${SKIP_LIST:-}" ] && grep -qx "$ds.$mem.$seed" "$SKIP_LIST" && return 0
  [ -s "$f.log" ] && [ "$(wc -l < "$f.log")" -ge 15 ] && return 0
  rm -f "$f.csv"
  ./bin/bench acc data/$ds.bin work/$ds.wl "$f.csv" "$ds" "$mem" "$seed" "$@" > "$f.log" 2> "$f.err"
}
export -f run_one
export OUT SKIP_LIST
printf '%s\n' "${jobs[@]}" | xargs -P "$J" -L 1 bash -c 'run_one $0 $1 $2 '"$METHODS"
echo "accuracy matrix done: $(ls $OUT/*.log | wc -l) jobs"


In [ ]:
%%writefile scripts/run_extra.sh
#!/usr/bin/env bash
# Ablations, attribute scaling (census), and timing. Run after run_accuracy.sh.
# Usage: scripts/run_extra.sh [parallel_jobs]
set -u
cd "$(dirname "$0")/.."
J=${1:-8}
mkdir -p results/abl results/census results/time

# ---------------------------------------------------------------- ablations (CoStrata design space)
ABL="cs:64:1:1 cs:256:1:1 cs:1024:1:1 cs:4096:1:1 cs:16384:1:1 cs:0:0:1 cs:0:0.25:1 cs:0:1:1 cs:0:4:1 cs:0:1e9:1 cs:0:0:0.5 cs:0:0:0.75 res"
run_abl() {
  ds=$1; mem=$2; seed=$3; shift 3
  f="results/abl/$ds.$mem.$seed"
  rm -f "$f.csv"
  ./bin/bench acc data/$ds.bin work/$ds.wl "$f.csv" "$ds" "$mem" "$seed" "$@" > "$f.log" 2> "$f.err"
}
export -f run_abl
for ds in taxi mawi zipf15_5M; do for mem in 1 10; do for seed in 1 2 3 4 5; do echo "$ds $mem $seed"; done; done; done |
  xargs -P "$J" -L 1 bash -c 'run_abl $0 $1 $2 '"$ABL"

# ---------------------------------------------------------------- attribute scaling on census (p = 4..68)
CEN="res uss omnip omni:64:2 omni:256:2 omni:1024:2 cs cs:0:1e9"
run_cen() {
  p=$1; seed=$2; shift 2
  f="results/census/census_p$p.10.$seed"
  rm -f "$f.csv"
  ./bin/bench acc data/census.bin work/census_p$p.wl "$f.csv" "census_p$p" 10 "$seed" -p "$p" "$@" > "$f.log" 2> "$f.err"
}
export -f run_cen
for p in 4 8 16 32 68; do
  [ -s work/census_p$p.wl ] || ./bin/mkworkload data/census.bin work/census_p$p.wl 200 4 7 $p 2> work/census_p$p.wl.log
done
for p in 4 8 16 32 68; do for seed in 1 2 3; do echo "$p $seed"; done; done |
  xargs -P 4 -L 1 bash -c 'run_cen $0 $1 '"$CEN"

# ---------------------------------------------------------------- timing (single pinned core, machine otherwise idle)
# Timings are only meaningful on a quiet, dedicated machine: set SKIP_TIMING=1 on shared VMs.
if [ -n "${SKIP_TIMING:-}" ]; then echo "extra experiments done (timing skipped)"; exit 0; fi
T=results/time/time.log
: > "$T"
for ds in taxi mawi zipf13_5M; do
  for mem in 1 10 50; do
    taskset -c 0 ./bin/bench time data/$ds.bin $ds $mem 1 3 res uss ussc omnip omnil omni:256:2 omni:1024:2 cs >> "$T"
  done
done
for p in 4 8 16 32 68; do
  taskset -c 0 ./bin/bench time data/census.bin census_p$p 10 1 3 -p $p res uss omnip omni:256:2 cs >> "$T"
done
# query latency (pinned, sequential)
Q=results/time/query.log
: > "$Q"
for ds in taxi mawi; do
  for mem in 1 10 50; do
    taskset -c 0 ./bin/bench acc data/$ds.bin work/$ds.wl /dev/null $ds $mem 1 res uss omnip omni:256:2 omni:1024:2 cs >> "$Q"
  done
done
echo "extra experiments done"


In [ ]:
%%writefile scripts/aggregate.py
#!/usr/bin/env python3
"""Aggregate per-query benchmark output into per-run metrics.

  aggregate.py <results_dir> <out.csv> [workload_dir]

For every (dataset, method, mem, seed) run we compute, per workload family:
  nae      mean |est - truth| / N            (the metric of the OmniSketch paper)
  mre      mean |est - truth| / truth
  mre_d{j} mean relative error over queries whose rarest predicate has frequency in [10^j, 10^(j+1))
  mre_k{k} mean relative error over queries with k predicates
  cov95    (CoStrata only) fraction of queries with |est - truth| <= 1.96 * sd
"""
import glob
import os
import sys

import numpy as np
import pandas as pd

res_dir, out = sys.argv[1], sys.argv[2]
wl_dir = sys.argv[3] if len(sys.argv) > 3 else "work"

workloads = {}


def workload(name):
    if name not in workloads:
        w = pd.read_csv(os.path.join(wl_dir, name + ".wl"), sep=" ", header=None, usecols=[0, 1, 2, 3],
                        names=["kind", "npred", "truth", "minfreq"], engine="c")
        workloads[name] = w
    return workloads[name]


rows = []
for log in sorted(glob.glob(os.path.join(res_dir, "*.log"))):
    base = log[:-4]
    if not os.path.exists(base + ".csv"):
        continue
    runs = {}
    for line in open(log):
        if line.startswith("RUN,"):
            f = line.rstrip("\n").split(",", 11)
            runs[f[2]] = dict(p=int(f[5]), n=int(f[6]), ns_per_rec=float(f[8]), query_us=float(f[9]),
                              used_mb=float(f[10]), info=f[11])
    df = pd.read_csv(base + ".csv", header=None, names=["ds", "method", "mem", "seed", "qid", "est", "ht", "sd", "cm"])
    wname = os.path.basename(base).split(".")[0]
    # p-scaling runs are named <dataset>_p<k>.<mem>.<seed>
    w = workload(wname)
    truth = w.truth.to_numpy(float)
    dec = np.floor(np.log10(np.maximum(w.minfreq.to_numpy(float), 1))).astype(int)
    for method, g in df.groupby("method", sort=False):
        if method not in runs:
            continue
        g = g.sort_values("qid")
        if len(g) != len(w):
            continue
        N = runs[method]["n"]
        est = g.est.to_numpy(float)
        err = np.abs(est - truth)
        rel = err / truth
        row = dict(dataset=g.ds.iloc[0], method=method, mem=float(g.mem.iloc[0]), seed=int(g.seed.iloc[0]), **runs[method])
        for kind in (0, 1):
            sel = (w.kind == kind).to_numpy()
            if sel.sum() == 0:
                continue
            row[f"nae{kind}"] = err[sel].mean() / N
            row[f"mre{kind}"] = rel[sel].mean()
            row[f"medre{kind}"] = np.median(rel[sel])
            for j in range(1, 7):
                s2 = sel & (dec == j)
                row[f"mre{kind}_d{j}"] = rel[s2].mean() if s2.sum() >= 5 else np.nan
                row[f"cnt{kind}_d{j}"] = int(s2.sum())
            for k in range(1, 7):
                s2 = sel & (w.npred == k).to_numpy()
                row[f"mre{kind}_k{k}"] = rel[s2].mean() if s2.sum() >= 5 else np.nan
                row[f"nae{kind}_k{k}"] = err[s2].mean() / N if s2.sum() >= 5 else np.nan
        row["nae_all"] = err.mean() / N
        if g.ht.notna().all():
            ht = g.ht.to_numpy(float)
            sd = g.sd.to_numpy(float)
            row["nae0_ht"] = np.abs(ht - truth)[(w.kind == 0).to_numpy()].mean() / N
            row["nae1_ht"] = np.abs(ht - truth)[(w.kind == 1).to_numpy()].mean() / N
            row["cov95"] = np.mean(err <= 1.96 * sd)
            big = truth >= 100
            row["cov95_big"] = np.mean(err[big] <= 1.96 * sd[big])
            row["bias_ht"] = ht.sum() / truth.sum() - 1.0
        rows.append(row)

pd.DataFrame(rows).to_csv(out, index=False)
print(f"wrote {out}: {len(rows)} runs")


In [ ]:
%%writefile scripts/convert.py
#!/usr/bin/env python3
"""Convert public datasets to the binary stream format used by the benchmarks.

  convert.py taxi   <yellow_tripdata_2019-01.parquet> <out.bin>
  convert.py census <USCensus1990.data.txt>           <out.bin>

Output: "STRM" | u32 p | u64 n | n*p u32 (row-major). All attributes are mapped
to non-negative 32-bit integers (equality semantics are preserved).
"""
import struct
import sys

import numpy as np


def write(path, cols, names):
    n, p = len(cols[0]), len(cols)
    arr = np.empty((n, p), dtype=np.uint32)
    for j, c in enumerate(cols):
        arr[:, j] = c
        print(f"  attr {j:2d} {names[j]:24s} distinct={len(np.unique(c))}")
    with open(path, "wb") as f:
        f.write(b"STRM")
        f.write(struct.pack("<IQ", p, n))
        arr.tofile(f)
    print(f"wrote {path}: n={n} p={p}")


def taxi(src, dst):
    import pyarrow.parquet as pq

    t = pq.read_table(src)
    g = lambda name: t.column(name).to_numpy(zero_copy_only=False)
    pick = g("tpep_pickup_datetime").astype("datetime64[s]").astype(np.int64)
    drop = g("tpep_dropoff_datetime").astype("datetime64[s]").astype(np.int64)
    dur_min = np.clip((drop - pick) // 60, 0, 600)
    hour = (pick // 3600) % 24
    dow = ((pick // 86400) + 4) % 7
    nz = lambda x: np.nan_to_num(x, nan=0.0)
    cols = [
        g("VendorID"),
        np.clip(nz(g("passenger_count")), 0, 15),
        np.clip(nz(g("RatecodeID")), 0, 120),
        g("PULocationID"),
        g("DOLocationID"),
        g("payment_type"),
        hour,
        dow,
        np.clip(np.round(nz(g("trip_distance")) * 10), 0, 5000),   # 0.1 mile bins
        np.clip(np.round(nz(g("fare_amount")) * 2), 0, 2000),      # $0.50 bins
        np.clip(np.round(nz(g("tip_amount"))), 0, 200),            # $1 bins
        dur_min,
    ]
    names = ["VendorID", "passenger_count", "RatecodeID", "PULocationID", "DOLocationID", "payment_type",
             "pickup_hour", "pickup_dow", "trip_distance_x10", "fare_x2", "tip_dollars", "duration_min"]
    # stream order = pickup time
    order = np.argsort(pick, kind="stable")
    write(dst, [np.asarray(c, dtype=np.int64)[order].astype(np.uint32) for c in cols], names)


def census(src, dst):
    with open(src) as f:
        names = f.readline().strip().split(",")[1:]
        data = np.loadtxt(f, delimiter=",", dtype=np.int64)
    data = data[:, 1:]  # drop caseid
    data = data - data.min(axis=0)
    write(dst, [data[:, j].astype(np.uint32) for j in range(data.shape[1])], names)


if __name__ == "__main__":
    {"taxi": taxi, "census": census}[sys.argv[1]](sys.argv[2], sys.argv[3])


In [ ]:
%%writefile colab/done_local.txt
zipf10_5M.0.5.1
zipf10_5M.0.5.2
zipf10_5M.0.5.3
zipf10_5M.0.5.4
zipf10_5M.0.5.5
zipf10_5M.10.1
zipf10_5M.10.2
zipf10_5M.10.3
zipf10_5M.10.4
zipf10_5M.10.5
zipf10_5M.1.1
zipf10_5M.1.2
zipf10_5M.1.3
zipf10_5M.1.4
zipf10_5M.1.5
zipf10_5M.20.1
zipf10_5M.20.2
zipf10_5M.20.3
zipf10_5M.20.4
zipf10_5M.20.5
zipf10_5M.2.1
zipf10_5M.2.2
zipf10_5M.2.3
zipf10_5M.2.4
zipf10_5M.2.5
zipf10_5M.50.1
zipf10_5M.50.2
zipf10_5M.50.3
zipf10_5M.50.4
zipf10_5M.50.5
zipf10_5M.5.1
zipf10_5M.5.2
zipf10_5M.5.3
zipf10_5M.5.4
zipf10_5M.5.5
zipf13_5M.0.5.1
zipf13_5M.0.5.2
zipf13_5M.0.5.3
zipf13_5M.0.5.4
zipf13_5M.0.5.5
zipf13_5M.10.1
zipf13_5M.10.2
zipf13_5M.10.3
zipf13_5M.10.4
zipf13_5M.10.5
zipf13_5M.1.1
zipf13_5M.1.2
zipf13_5M.1.3
zipf13_5M.1.4
zipf13_5M.1.5
zipf13_5M.20.1
zipf13_5M.2.1
zipf13_5M.2.2
zipf13_5M.2.3
zipf13_5M.2.4
zipf13_5M.2.5
zipf13_5M.5.1
zipf13_5M.5.2
zipf13_5M.5.3
zipf13_5M.5.4
zipf13_5M.5.5


In [ ]:
%%bash
set -e
cd /content/costrata
chmod +x scripts/*.sh
make 2>&1 | tail -5
ls -la bin

In [ ]:
%%bash
set -e
cd /content/costrata
# synthetic streams (generator and seeds identical to the local run)
[ -s data/zipf10_5M.bin ] || ./bin/gen_synth data/zipf10_5M.bin 5000000 5 1.0 10000 1
[ -s data/zipf13_5M.bin ] || ./bin/gen_synth data/zipf13_5M.bin 5000000 5 1.3 10000 2
[ -s data/zipf15_5M.bin ] || ./bin/gen_synth data/zipf15_5M.bin 5000000 5 1.5 10000 3
[ -s data/unif_5M.bin ]   || ./bin/gen_synth data/unif_5M.bin   5000000 5 0   10000 4
# NYC yellow taxi, January 2019
if [ ! -s data/taxi.bin ]; then
  curl -sL -o data/raw/yellow_2019-01.parquet https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet
  python scripts/convert.py taxi data/raw/yellow_2019-01.parquet data/taxi.bin
fi
# MAWI backbone trace 2010-01-01 14:00 (samplepoint F)
if [ ! -s data/mawi.bin ]; then
  curl -sL -o data/raw/mawi_201001011400.dump.gz http://mawi.wide.ad.jp/mawi/samplepoint-F/2010/201001011400.dump.gz
  ./bin/pcap2bin data/raw/mawi_201001011400.dump.gz data/mawi.bin
fi
# US Census 1990 (UCI)
if [ ! -s data/census.bin ]; then
  curl -sL -o data/raw/census1990.zip "https://archive.ics.uci.edu/static/public/116/us+census+data+1990.zip"
  unzip -o -q data/raw/census1990.zip USCensus1990.data.txt -d data/raw
  python scripts/convert.py census data/raw/USCensus1990.data.txt data/census.bin | tail -1
fi
for ds in zipf10_5M zipf13_5M zipf15_5M unif_5M; do [ -s work/$ds.wl ] || ./bin/mkworkload data/$ds.bin work/$ds.wl 200 5 7 2>&1 | tail -1; done
for ds in taxi mawi; do [ -s work/$ds.wl ] || ./bin/mkworkload data/$ds.bin work/$ds.wl 200 6 7 2>&1 | tail -1; done
ls -la data/*.bin work/*.wl

In [ ]:
import hashlib
LOCAL_MD5 = {
 "data/census.bin": "e69071ef0ba0b78716727c035aa62217",
 "data/mawi.bin": "7491f634d22289f00992a4cd3aec9570",
 "data/taxi.bin": "2c5a25834c3573172bf934d0928e357a",
 "data/unif_5M.bin": "a493edffacb10298843df8bfb41a63e0",
 "data/zipf10_5M.bin": "82fdc8845ed2df2d518c705c594a07f0",
 "data/zipf13_5M.bin": "5b312733dd299abf6b16ffca707abb95",
 "data/zipf15_5M.bin": "13d59487418153094e2910e58f48e530",
 "work/zipf10_5M.wl": "133b14d9b8dfbf3ecba910103b85f094",
 "work/zipf13_5M.wl": "d0c20e89858360ba05c8b950916708f8",
 "work/zipf15_5M.wl": "5d248bacc3b808fd2b7448d16a0dcea9",
 "work/unif_5M.wl": "4b33eb876e198624bc0b89cecb598a80",
 "work/taxi.wl": "d98b5860200fc456feb35552997ce6c3",
 "work/mawi.wl": "9645b40737930fa7448e0c677a1b3fd3"
}
def md5(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 22), b''):
            h.update(chunk)
    return h.hexdigest()
same = True
for path, want in LOCAL_MD5.items():
    got = md5(path)
    ok = got == want
    if not ok and 'census' not in path:
        same = False
    print(('OK      ' if ok else 'DIFFERS ') + path)
SKIP_LIST = f'{ROOT}/colab/done_local.txt' if same else ''
print('\nData identical to the local run -> skipping the 61 jobs already finished there.' if same
      else '\nData differs from the local run -> running the full matrix here (local partial results must then be discarded).')
open(f'{DRIVE}/data_identical.txt', 'w').write(str(same))

In [ ]:
# restore earlier progress from Drive (if the runtime was recycled), then launch everything in the background
subprocess.run(f'rsync -a {DRIVE}/results/ {ROOT}/results/', shell=True)
env = dict(os.environ, SKIP_LIST=SKIP_LIST, SKIP_TIMING='1')
J = os.cpu_count()
log = open(f'{ROOT}/run.out', 'w')
proc = subprocess.Popen(['bash', '-c', f'bash scripts/run_accuracy.sh {J} && bash scripts/run_extra.sh {J}'],
                        cwd=ROOT, env=env, stdout=log, stderr=subprocess.STDOUT)
print('started, pid', proc.pid, 'parallel jobs:', J)

In [ ]:
# progress monitor: mirrors results to Drive every 2 minutes; keep this cell running until it prints DONE
import glob, time
def complete(d, need):
    n = 0
    for f in glob.glob(f'{ROOT}/results/{d}/*.log'):
        with open(f) as fh:
            n += sum(1 for _ in fh) >= need
    return n
t0 = time.time()
while True:
    done = proc.poll() is not None
    subprocess.run(f'rsync -a {ROOT}/results/ {DRIVE}/results/', shell=True)
    print(f'[{(time.time() - t0) / 60:6.1f} min] accuracy {complete("acc", 15)}/210 (61 may be skipped) | '
          f'ablation {complete("abl", 13)}/30 | census {complete("census", 8)}/15', flush=True)
    if done:
        break
    time.sleep(120)
print('DONE, exit code', proc.returncode)
print(open(f'{ROOT}/run.out').read()[-2000:])

In [ ]:
%%bash
cd /content/costrata
python scripts/aggregate.py results/acc results/agg_acc.csv
python scripts/aggregate.py results/abl results/agg_abl.csv
python scripts/aggregate.py results/census results/agg_census.csv
cat results/*/*.err 2>/dev/null | head -20
D=/content/drive/MyDrive/costrata_results
cp results/agg_*.csv $D/
cat results/acc/*.log results/abl/*.log results/census/*.log > $D/run_lines.csv
md5sum data/*.bin work/*.wl > $D/md5_colab.txt
ls -la $D

When the last cell has finished, `MyDrive/costrata_results/` contains `agg_acc.csv`, `agg_abl.csv`, `agg_census.csv`
(a few hundred KB in total) plus the raw per-query output under `results/`. Only the three `agg_*.csv` files,
`run_lines.csv` and `data_identical.txt` are needed to finish the paper.